# MICrONS PCA — Option 2: Trial-averaged PCA across cortical areas

Applying PCA per cortical area to trial-averaged responses, asking whether the three stimulus classes (Clip, Monet2, Trippy) occupy distinct regions of population state space and whether the strength of that separation differs across areas (V1, AL, LM, RL).

See `docs/specs/2026-05-02-pca-design.md` for the design and `docs/plans/2026-05-02-pca-implementation.md` for the implementation plan.

## How to use this notebook

The pipeline is **session-swappable**: every session-specific value is read from data, and outputs are partitioned by session subdirectory. To run on a different session, change `SESSION` in the configuration block below and restart the kernel.

## Part 0 — Setup

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.io as pio

import microns_eda
import option2_pca_utils as pca_utils

In [ ]:
# === Configuration block — every tunable lives here. ===
SESSION = "7_5"

# Preprocessing
N_FRAMES_TRUNCATE = 75              # Clip trial length; truncates Monet2/Trippy from 113.
TREADMILL_OUTLIER_THRESHOLD = 1.0   # Same threshold as EDA's running-outlier filter.

# PCA / metrics
N_COMPONENTS = 10
N_BALANCE_REPLICATES = 20           # subsamples per balanced silhouette.
N_SHUFFLES = 100                    # for both silhouette and classifier nulls.
N_POPULATION_SUBSAMPLES = 20        # for the equal-population control.
N_FOLDS_CV = 5

# Reproducibility
RANDOM_SEED = 42

# Paths
DATADIR = Path(os.environ.get("MICRONS_DATADIR", "../neuroscience"))
FIGURES_DIR = Path(f"figures/option2/{SESSION}")
RESULTS_DIR = Path(f"results/option2/{SESSION}")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)

print(f"SESSION       = {SESSION}")
print(f"DATADIR       = {DATADIR}")
print(f"FIGURES_DIR   = {FIGURES_DIR}")
print(f"RESULTS_DIR   = {RESULTS_DIR}")

### Sanity check

We open the dataset, verify session `{SESSION}` is loadable, recompute `clean_trial_indices` from the EDA's running-outlier rule, and print the per-class composition. **All session-specific values are read from data** — switching `SESSION` changes the printed numbers but no code.

This cell fails loudly if the environment is wrong (path, library versions, missing session). Pass = safe to proceed.

In [ ]:
# === Sanity check ===
reader = microns_eda.open_dataset(DATADIR)
sessions = microns_eda.list_sessions(DATADIR)
assert SESSION in sessions, f"session {SESSION!r} not in {sessions}"
print(f"Found {len(sessions)} sessions; using {SESSION!r}.")

meta = microns_eda.get_session_meta(DATADIR, SESSION)
print(f"  n_neurons (read from data) = {meta['n_neurons']}")

# Build per-trial treadmill means via the same path as EDA.
n_trials = meta["n_trials"]
per_trial_tread_means = np.empty(n_trials, dtype=np.float64)
for i in range(n_trials):
    trial = microns_eda.load_trial(reader, DATADIR, SESSION, i)
    per_trial_tread_means[i] = np.nanmean(trial["treadmill"])

clean_trial_indices, running_mask = microns_eda.compute_clean_trial_indices(
    per_trial_tread_means, threshold=TREADMILL_OUTLIER_THRESHOLD,
)

# Stim labels for the clean subset.
stim_map = microns_eda.build_stim_type_map(reader, SESSION)
stim_types_per_trial = np.array([
    stim_map[h.decode() if isinstance(h, bytes) else h]
    for h in meta["condition_hashes"]
])
labels = stim_types_per_trial[clean_trial_indices]

# Read class counts from data — never hard-code.
class_counts = pd.Series(labels).value_counts().to_dict()
print(f"  n_trials (total)            = {n_trials}")
print(f"  n_trials (after running QC) = {len(clean_trial_indices)}")
print(f"  per-class breakdown (clean):")
for k in sorted(class_counts):
    print(f"    {k:8s} {class_counts[k]}")

assert len(clean_trial_indices) > 0, "No clean trials remain — threshold too low?"
assert len(class_counts) >= 2, "Need at least 2 stim classes for separability."
print("Sanity check passed.")